# 4 · Flow fields & networks

Two families: **u/v vector fields** (`vectorfield` arrows, `streamlines`, `barbs`) and **node–edge
networks** (`graph`, `flow`). For the field we use the Lisbon DEM's downhill gradient as a real
`(u, v)` pair; for the network a small synthetic station graph.

**Setup** — derive a coarse `(u, v)` field from the DEM gradient (downsampled so the arrows stay legible).

In [ ]:
from pathlib import Path

# Resolve the repo root so the bundled sample data is found whether this runs from
# docs/examples/interactive/ (mkdocs) or the repository root.
ROOT = Path.cwd()
while not (ROOT / "examples" / "data" / "LisbonElevation.tif").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "examples" / "data"

import holoviews as hv
hv.extension("bokeh")            # the interactive tier renders through Bokeh

import numpy as np
from pyramids.dataset import Dataset
from digitalearth.interactive import InteractiveMap

dem = Dataset.read_file(str(DATA / "LisbonElevation.tif"))
coarse = dem.resample(dem.cell_size * 12)                # fewer, legible arrows
z = np.nan_to_num(coarse.read_array(band=0).astype("float32"))
dy, dx = np.gradient(z)
u = Dataset.create_from_array(arr=(-dx).astype("float32"), geo=coarse.geotransform, epsg=coarse.epsg)
v = Dataset.create_from_array(arr=( dy).astype("float32"), geo=coarse.geotransform, epsg=coarse.epsg)

### `vectorfield` — arrows coloured by magnitude
Interactive arrows; `color_by='magnitude'` (default) colours each by speed. `density` thins the grid (`1.0` keeps every cell).

In [ ]:
m = InteractiveMap(crs=dem.epsg, title="downhill gradient (vectorfield)")
m.vectorfield(u, v, density=0.6, cmap="viridis")
m

### `streamlines` & `barbs` — matplotlib-backend flow
Bokeh has no streamline/wind-barb glyph, so these render through HoloViews' **matplotlib** backend (a static layer — save to `.png`, not `.html`). They are logged as non-interactive rather than emitting an empty Bokeh layer. `barbs` also takes a `density`.

In [ ]:
hv.extension("matplotlib")                 # streamlines/barbs render through matplotlib
m = InteractiveMap(crs=dem.epsg, title="streamlines")
m.streamlines(u, v, density=0.5)
hv.output(m.render(), backend="matplotlib")

In [ ]:
m = InteractiveMap(crs=dem.epsg, title="wind barbs")
m.barbs(u, v, density=0.4)
hv.output(m.render(), backend="matplotlib")

### `graph` / `flow` — node–edge networks
`graph` draws nodes (a point GeoDataFrame) joined by edges (pairs of node ids, optionally weighted); `flow` is the same with flow-styled edges. Notebooks may build a GeoDataFrame directly (the DX.3 pyramids-only rule applies to the package, not to example notebooks).

In [ ]:
hv.extension("bokeh")
import geopandas as gpd

nodes = gpd.GeoDataFrame(
    {"id": [0, 1, 2, 3]},
    geometry=gpd.points_from_xy([7.0e5, 7.6e5, 8.2e5, 7.6e5], [6.25e6, 6.35e6, 6.30e6, 6.45e6]),
    crs="EPSG:3857",
)
edges = [(0, 1, 5.0), (1, 2, 3.0), (2, 3, 2.0)]

m = InteractiveMap(crs=3857, title="station network (graph)")
m.graph(nodes, edges, weight="weight")
m